# 15. Empaquetado, inferencia y despliegue (API)

**Fases del guía metodológica cubiertas: 21 (Empaquetado, inferencia y despliegue)**



## 21.1 Pipeline de inferencia

```
Input crudo (JSON validado por Pydantic)
-> validación de esquema (tipos, rangos, categorías)
-> preprocessor entrenado (OneHot + Ordinal + mediana) — el MISMO objeto guardado
-> feature engineering (add_domain_features)
-> modelo RandomForest
-> umbral de coste congelado
-> respuesta {probability, prediction, label}
```

El sistema de inferencia usa **exactamente** el pipeline guardado en
`models/final_model.joblib`; no se recrea "similar" (regla de oro de ML en producción).

### 21.1.1 Prueba del pipeline completo (JSON -> predicción)

Simulamos el flujo real de la API: partimos de un JSON con las 29 columnas originales,
reconstruimos las features derivadas (igual que `add_domain_features`) y aplicamos el
pipeline guardado. La probabilidad resultante se compara con el umbral congelado para
emitir la clase. Esta celda es la prueba de que el contrato entrenamiento-producción
funciona de extremo a extremo.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Prueba del pipeline completo: de JSON crudo a predicción
import joblib, json, pandas as pd
from src.features.build_features import add_domain_features

pipeline = joblib.load(ROOT / "models" / "final_model.joblib")
meta = json.loads((ROOT / "models" / "final_model_metadata.json").read_text(encoding="utf-8"))

def json_a_features(record):
    # Convierte el JSON del usuario en el DataFrame con las features esperadas.
    df = pd.DataFrame([record])
    df["family_support"] = (df["schoolsup"] == "yes").astype(int) + (df["famsup"] == "yes").astype(int)
    df["study_intensity"] = df["studytime"] + df["paid"].map({"yes": 1, "no": 0})
    df["has_failures"] = (df["failures"] > 0).astype(int)
    df["absences_high"] = (df["absences"] >= 8).astype(int)
    df["parent_edu_max"] = df[["Medu", "Fedu"]].max(axis=1)
    df["parent_edu_diff"] = (df["Medu"] - df["Fedu"]).abs()
    df["social_exposure"] = df["goout"] + df["freetime"]
    df["health_low"] = (df["health"] <= 2).astype(int)
    return df[meta["features"]]

ejemplo = {"school": "GP", "sex": "M", "age": 17, "address": "U", "famsize": "GT3",
           "Pstatus": "A", "Medu": 3, "Fedu": 3, "Mjob": "other", "Fjob": "other",
           "reason": "course", "guardian": "mother", "traveltime": 1, "studytime": 2,
           "failures": 0, "schoolsup": "no", "famsup": "yes", "paid": "no",
           "activities": "no", "nursery": "yes", "higher": "yes", "internet": "yes",
           "romantic": "yes", "famrel": 4, "freetime": 3, "goout": 5, "Dalc": 2,
           "health": 4, "absences": 4}
X = json_a_features(ejemplo)
proba = pipeline.predict_proba(X)[0, 1]
print("Probabilidad de consumo alto:", round(proba, 4),
      "->", "ALTO" if proba >= meta["threshold"] else "bajo",
      f"(umbral {meta['threshold']:.2f})")


Probabilidad de consumo alto: 0.9131 -> ALTO (umbral 0.34)



## 21.2 Formas de entrega

1. **API REST (FastAPI)** — `src/api/main.py`, endpoints `/predict`, `/predict_batch`,
   `/features`, `/health`. Documentación interactiva en `/docs`.
2. **Script CLI** — `python scripts/run_pipeline.py` (pipeline completo de principio a fin).
3. **Batch** — `scripts/batch_inference.py` para CSV.
4. **Notebook reproducible** — este proyecto.

## 21.3 Validación de entradas

La API valida: esquema (campos obligatorios), tipos, rangos (edad 10-22, ordinales 1-5),
categorías permitidas (Pydantic `Literal`), y política de abstención (zona 0.30-0.60).

### 21.3.1 Prueba de los endpoints con TestClient

Levantamos la aplicación FastAPI en memoria (TestClient) y probamos: una predicción
válida (200), una predicción con un ordinal fuera de rango (422, rechazada por Pydantic)
y el listado de features. Esto verifica que la validación de inputs funciona antes de
desplegar.


In [2]:

# Prueba de los endpoints sin levantar el servidor (TestClient de FastAPI)
from fastapi.testclient import TestClient
from src.api.main import app
client = TestClient(app)

r = client.post("/predict", json=ejemplo)
print("POST /predict ->", r.status_code, r.json())

r_bad = client.post("/predict", json={**ejemplo, "goout": 99})
print("POST /predict con goout=99 ->", r_bad.status_code, r_bad.json()["detail"][0]["msg"][:60])


C:\Users\sgml1\Desktop\student-alcohol-consumption\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


POST /predict -> 200 {'probability': 0.8812, 'prediction': 1, 'label': 'alto', 'threshold': 0.33999999999999997, 'features_used': 29}
POST /predict con goout=99 -> 422 Input should be less than or equal to 5



### Cómo levantar la API

```bash
uvicorn src.api.main:app --reload --port 8000
# Documentación: http://127.0.0.1:8000/docs
# Prueba:        http://127.0.0.1:8000/health
```

### Docker (opcional)

```dockerfile
FROM python:3.11-slim
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
CMD ["uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"]
```
